# Additional manoeuvre data: frozen NN pilots versus NN pilots

Acquire **10,000 new physical engagements**, with **20,000 recorded aircraft perspectives**.
Both F-16s use the validation-selected, post-tactical-adaptation NN checkpoints reported
in [07_tactical_adaptation.ipynb](07_tactical_adaptation.ipynb). There are no PPO updates.
Every engagement has a fresh continuous relative geometry and seed. Flights stop at
**300 in-game seconds or the first aircraft defeat**, including a crash. Both pilots'
51-feature observations and physical tracks are recorded at **10 Hz**, and their actual
selected skills/parameters at **1 Hz**. All outcomes are retained.

**Run All downloads the verified pilot archive, benchmarks CPU and GPU-batched rollouts, collects the
campaign, verifies it, and commits/pushes incremental data backups to this branch with
Git LFS.** Set `COLLECT=False` and `PUBLISH=False` to inspect the setup first. Rerunning
resumes the same campaign; it never counts a second perspective as another flight.
The notebook is a long-running data acquisition job. Its benchmark estimates the actual
wall time and storage requirement before the main run. The original corpora remain usable.

Install the repository's dependencies in this kernel first: `%pip install -e ".[ml,video]"`.
Git and Git LFS must be installed, and `forecasting_ed_261104` must be checked out for
publication. Use `GIT_EXECUTABLE` below on Windows if Git is absent from the kernel PATH.
The companion script runs in fresh spawned processes on Windows and Linux.

**GPU update:** one inference process batches all ten independent actors across CPU simulator workers. Run All selects the measured fastest backend; CUDA needs a working CUDA-enabled PyTorch install. No 10,000-flight job has been run in the cloud. The new default work/output paths keep previous collections separate.

In [ ]:
%%time
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
import numpy as np
import pandas as pd
import torch
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "scripts/generate_nn_duel_corpus.py").exists())
sys.path.insert(0, str(ROOT / "scripts"))
import generate_nn_duel_corpus as duel
from nn_duel_progress import NotebookProgress

FLIGHTS = int(os.getenv("BVR_NN_FLIGHTS", "10000"))
SEED = int(os.getenv("BVR_NN_SEED", "26110600"))
EPISODE_SECONDS = float(os.getenv("BVR_NN_SECONDS", "300"))
WORK = Path(os.getenv("BVR_NN_WORK", str(ROOT / "artifacts/nn_duels_batched_v1"))).resolve()
OUTPUT = Path(os.getenv("BVR_NN_OUTPUT", str(ROOT / "shared/corpora/nn-duels-batched-v1"))).resolve()
DOWNLOAD_LFS = os.getenv("BVR_NN_DOWNLOAD", "1") == "1"
RUN_BENCHMARK = os.getenv("BVR_NN_BENCHMARK", "1") == "1"
COLLECT = os.getenv("BVR_NN_COLLECT", "1") == "1"
PUBLISH = os.getenv("BVR_NN_PUBLISH", "1") == "1"
GIT_EXECUTABLE = None  # e.g. r"C:\Program Files\Git\cmd\git.exe"
DETERMINISTIC = True
CUDA_AVAILABLE = torch.cuda.is_available()
BACKEND = os.getenv("BVR_NN_BACKEND", "auto")  # auto, central_cuda, local_cpu; central_cpu is for IPC tests.
BUDGET_WORKERS = max(1, min(32, int((os.cpu_count() or 1) * .8)))
WORKER_CANDIDATES = sorted({max(1, BUDGET_WORKERS//2), BUDGET_WORKERS})
BENCHMARK_FLIGHTS = int(os.getenv("BVR_NN_BENCHMARK_FLIGHTS", str(max(30, 2*BUDGET_WORKERS))))
WORKERS = BUDGET_WORKERS
BATCH_WAIT_MS = 2.0
BATCH_MAX_REQUESTS = 32
BACKUP_EVERY = 100
SHARD_FLIGHTS = 100
UPLOAD_MBPS = float(os.getenv("BVR_NN_UPLOAD_MBPS", "20"))  # Assumption; enter your sustained upload rate.
CHECKSUM_READ_MBPS = float(os.getenv("BVR_NN_SCAN_MBPS", "400"))  # Decimal MB/s, including cache effects.
GIT_SECONDS_PER_SNAPSHOT = 10.0
PROGRESS_INTERVAL_SECONDS = float(os.getenv("BVR_NN_PROGRESS_SECONDS", "300"))
FULLY_VERIFIED = False
SCRIPT = ROOT / "scripts/generate_nn_duel_corpus.py"
WORK.mkdir(parents=True, exist_ok=True)
if BACKEND == "central_cuda" and not CUDA_AVAILABLE:
    raise RuntimeError("CUDA requested but unavailable. Install CUDA-enabled PyTorch or choose local_cpu/auto.")

runtime = {"logical_cpus": os.cpu_count(), "torch": str(torch.__version__),
           "torch_cuda_runtime": torch.version.cuda, "cuda_available": CUDA_AVAILABLE}
if CUDA_AVAILABLE:
    properties = torch.cuda.get_device_properties(0)
    runtime.update(gpu=properties.name, gpu_memory_gib=properties.total_memory/2**30,
                   compute_capability=str(torch.cuda.get_device_capability(0)))
    probe = torch.ones((16,16),device="cuda")
    probe = probe @ probe
    torch.cuda.synchronize()
    del probe
    runtime["cuda_kernel_check"] = "passed"
display(pd.Series(runtime, name="Detected execution hardware"))

updates = NotebookProgress(WORK, FLIGHTS, interval_seconds=PROGRESS_INTERVAL_SECONDS)

def run(command, *, work=WORK, output=OUTPUT, flights=FLIGHTS, seed=SEED,
        workers=None, backend=None, limit=None, full_verify=False):
    backend = backend or ("central_cuda" if BACKEND == "auto" and CUDA_AVAILABLE else
                          "local_cpu" if BACKEND == "auto" else BACKEND)
    args = [sys.executable, "-u", str(SCRIPT), command, "--work", str(work),
            "--output", str(output), "--flights", str(flights), "--seed", str(seed),
            "--seconds", str(EPISODE_SECONDS), "--workers", str(workers or WORKERS),
            "--shard-flights", str(SHARD_FLIGHTS), "--inference-backend", backend,
            "--batch-wait-ms", str(BATCH_WAIT_MS), "--batch-max-requests", str(BATCH_MAX_REQUESTS)]
    if limit is not None:
        args += ["--limit", str(limit)]
    if not DETERMINISTIC:
        args += ["--stochastic"]
    if command in ("verify", "publish"):
        args += ["--verification-cache", str(Path(work) / "verified_snapshot.json")]
    if command == "publish":
        args += ["--allow-partial"]
    env = os.environ.copy()
    if GIT_EXECUTABLE:
        env["PATH"] = str(Path(GIT_EXECUTABLE).parent) + os.pathsep + env.get("PATH", "")
    for key in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
        env[key] = "1"
    if full_verify:
        args = [sys.executable, "-u", str(ROOT / "scripts/nn_duel_progress.py"),
                "verify-complete", "--output", str(output)]
    stage = {"prepare": "Preparing frozen pilots and scenarios",
             "collect": "Collecting flights", "package": "Packaging saved flights",
             "verify": "Verifying saved data", "publish": "Verifying and uploading Git LFS backup"}[command]
    if command == "collect" and Path(work).resolve() != WORK:
        stage = f"Benchmark: {backend}, {workers or WORKERS} simulator workers"
    if full_verify:
        stage = "Final full verification"
    updates.run(args, cwd=ROOT, env=env, command=command, work=work, stage=stage, target=flights)

print(f"Target: {FLIGHTS:,} physical flights / {2*FLIGHTS:,} perspectives; <= {EPISODE_SECONDS}s")
print("Worker candidates:", WORKER_CANDIDATES, "| requested inference:", BACKEND)
print("Work:", WORK, "| LFS publication:", OUTPUT)
print(f"Human-readable progress about every {PROGRESS_INTERVAL_SECONDS/60:g} minutes; raw command logs: {updates.logs}")

## Frozen population and actual initial conditions

The source is the **5.7 MB `reproduction.zip` in `trained-pilot-episodes-v1`**,
verified against its publication manifest. It contains all ten *post*-tactical
checkpoints, their architectures/configuration and the source/assets used for that
corpus. The notebook-05 public checkpoint pool predates tactical adaptation and is
not substituted. Each source file and checkpoint hash is checked before collection;
source, software and configuration must match for resumption.

The old spawn adapter used a sampled axis, opposing courses and fixed speeds; nominal
scenario bearings/headings were not actually applied. Here the requested position,
heading and speed are passed directly into aircraft creation during reset, so JSBSim
starts in that geometry. The collector checks actual reset positions and velocity
vectors. North/West/Up mathematical angles are converted into the proper velocity
vectors; the original guidance interface retains its own heading convention.

Each continuous dimension uses stratified samples. Ten families cover mixed aspects,
left/right crossings, tail chase, cold/recommit, close merges, low energy climbs,
high/low encounters, offset support and defensive beam geometries. Horizontal
separation spans 6–65 km, altitude 1.5–11.5 km, and speed 200–380 m/s.
Initial velocities exclude the old corpus's exactly antiparallel pattern; all
requested states are unique. Novel geometry is not a guarantee of novel NN behaviour.
The 100 ordered pilot pairs, including self-pairs, each receive 100 fresh flights
in the default campaign. Roles and pair order are balanced without reusing geometry.

Frozen package imports are pinned in each child process, including when this repository
is installed in editable mode. Failed benchmark candidates are retried on rerun.
Setup and benchmark directories are keyed by collector source hashes. If an older
main campaign already has `WORK/config.json`, choose a fresh `BVR_NN_WORK` and
`BVR_NN_OUTPUT` before running this revised collector; source changes cannot resume
an immutable campaign. Previously published data remains verifiable.


In [ ]:
%%time
if DOWNLOAD_LFS:
    # Reuse 09b's Windows-aware Git discovery, but download only the small pilot archive.
    sys.path.insert(0, str(ROOT))
    from bvr_behavior_prediction.paired_skill_forecasting import download_paired_corpora
    download_paired_corpora(ROOT, ["shared/corpora/trained-pilot-episodes-v1/reproduction.zip"],
                           git_executable=GIT_EXECUTABLE)
# Provision the frozen archive and geometry without fixing the main campaign backend yet.
setup_sources = {p.name: duel.sha256(p) for p in duel.collector_sources()}
setup_id = hashlib.sha256(json.dumps(setup_sources, sort_keys=True).encode()).hexdigest()[:12]
SETUP_WORK = WORK / "setup" / setup_id
run("prepare", work=SETUP_WORK, backend="local_cpu")
config = duel.read(SETUP_WORK / "config.json")
geometry = pd.DataFrame([{
    "family": s["family"], "pilot_A01": s["pilots"]["A01"], "pilot_B01": s["pilots"]["B01"],
    "horizontal_separation_m": np.linalg.norm(np.array(s["initial"]["B01"]["position"][:2]) -
                                             np.array(s["initial"]["A01"]["position"][:2])),
    "height_offset_m": s["initial"]["B01"]["position"][2] - s["initial"]["A01"]["position"][2],
    "heading_difference_deg": (s["initial"]["B01"]["heading_nwu_deg"] - s["initial"]["A01"]["heading_nwu_deg"]) % 360,
} for s in config["cases"]])
display(geometry.groupby("family").agg(flights=("family", "size"),
    minimum_range_m=("horizontal_separation_m", "min"), maximum_range_m=("horizontal_separation_m", "max"),
    minimum_height_offset_m=("height_offset_m", "min"), maximum_height_offset_m=("height_offset_m", "max")))
display(geometry.groupby(["pilot_A01", "pilot_B01"]).size().unstack(fill_value=0))
display(pd.DataFrame(config["pilots"]).T)

## Shared GPU actor batching and measured throughput

CPU processes retain independent JSBSim worlds, RNGs and two-second histories. With
`central_cuda`, **one additional process owns the CUDA context and all ten frozen
actors**. Workers send both aircraft histories over CPU pipes once per game second.
A configurable **2 ms maximum batching window** combines ready requests. `torch.func.vmap`
evaluates the independent actor weights together in one stacked forward; policies are
never averaged. Padding has no effect on returned actions. Only the actor encoder,
skill logits and parameter means run; unused critic and log-probability calculations
are omitted. The owner returns each selected skill and its original decoded parameters
in request order. Simulator workers never initialize CUDA or copy a model to the GPU.

This is deterministic FP32 inference with TF32 disabled. Every owner startup compares
80 histories across all ten actors against CPU reference logits/means and requires the
same selected skills, with `2e-5` numeric tolerances. This checks the actor math, not
bit-identical trajectories: GPU arithmetic and changing batch shapes can lead to small
differences that accumulate in combat. Source/precision/backend are recorded in the
immutable configuration. Changing backend requires a new campaign. Stochastic-policy
collection remains available with `local_cpu`; centralized batching does not silently
change stochastic sampling semantics.

A small cloud CPU profile put NN decisions at **12.5%** of rollout wall time: ideal
zero-cost inference would improve throughput by about **14%**. Most work is CPU physics,
sensors and missiles. GPU batching may therefore provide a modest gain or lose to IPC,
padding and launch costs. The benchmark compares **whole-flight throughput**, including
process/model startup, against local CPU inference, and selects the fastest successful
backend. Set `BACKEND="central_cuda"` to force the GPU option explicitly. CUDA failures
are shown in the results table; auto mode can retain a working CPU baseline.

The default suite has at least 30 full-duration-limited flights and at least two per
worker. Increase it to 100+ for a steadier estimate. All candidates use the same seeds,
pairs and starting geometry. Benchmark flights are separate from the main 10,000.
Unused rewards and scripted expert labels, videos and ACMI remain disabled. Physics
cadence and the native forced-throttle behaviour are unchanged. All prior payload bytes
are still checksummed before publication, while unchanged per-episode semantic checks
use a receipt keyed by the actual payload hashes, configuration and episode index row.
A final complete verification does not use that receipt.

In [ ]:
%%time
benchmark_path = WORK / "benchmark_batched.json"
backends = (["local_cpu", "central_cuda"] if CUDA_AVAILABLE and DETERMINISTIC else ["local_cpu"]) if BACKEND == "auto" else [BACKEND]
benchmark_identity = {"collector_hashes": config["collector_hashes"], "software": config["software"],
    "runtime": runtime, "workers": WORKER_CANDIDATES, "backends": backends,
    "flights": BENCHMARK_FLIGHTS, "seconds": EPISODE_SECONDS, "deterministic": DETERMINISTIC,
    "batch_wait_ms": BATCH_WAIT_MS, "batch_max_requests": BATCH_MAX_REQUESTS}
if RUN_BENCHMARK:
    cached = duel.read(benchmark_path) if benchmark_path.exists() else None
    # Reuse successful measurements, but retry failed candidates after a runtime repair.
    reusable = {(row["backend"], row["workers"]): row for row in cached["results"]
                if row["status"] == "ok"} if cached and cached["identity"] == benchmark_identity else {}
    rows = []
    bench_id = hashlib.sha256(json.dumps(benchmark_identity, sort_keys=True).encode()).hexdigest()[:12]
    for backend in backends:
        for workers in WORKER_CANDIDATES:
            if (backend, workers) in reusable:
                rows.append(reusable[(backend, workers)])
                continue
            bench_work = WORK / "benchmarks" / bench_id / f"{backend}_{workers}"
            try:
                run("collect", work=bench_work, flights=BENCHMARK_FLIGHTS, seed=SEED+100000,
                    workers=workers, backend=backend)
                summaries = [duel.completed(bench_work, s) for s in duel.read(bench_work / "config.json")["cases"]]
                progress = duel.read(bench_work / "progress.json")
                elapsed = progress.get("cumulative_wall_seconds", progress["wall_seconds"])
                signatures = sorted((s["episode_id"], sorted(s["files"].items())) for s in summaries)
                post_start = time.perf_counter()
                run("package", work=bench_work, output=bench_work / "published", flights=BENCHMARK_FLIGHTS,
                    seed=SEED+100000, backend=backend)
                run("verify", work=bench_work, output=bench_work / "published", backend=backend)
                post_seconds = time.perf_counter()-post_start
                inference = duel.read(bench_work / "inference_metrics.json") if (bench_work / "inference_metrics.json").exists() else {}
                rows.append({"backend": backend, "workers": workers, "status": "ok",
                    "wall_seconds": elapsed, "flights_per_hour": BENCHMARK_FLIGHTS*3600/elapsed,
                    "postprocess_seconds": post_seconds,
                    "mean_game_seconds": np.mean([s["duration_s"] for s in summaries]),
                    "decision_fraction": sum(s["timing"]["decision_seconds"] for s in summaries)/sum(s["wall_seconds"] for s in summaries),
                    "bytes_per_flight": sum(p.stat().st_size for p in (bench_work / "episodes").rglob("*.parquet"))/BENCHMARK_FLIGHTS,
                    "mean_actor_rows_per_gpu_dispatch": inference.get("mean_rows_per_dispatch", 0),
                    "data_signature": hashlib.sha256(json.dumps(signatures).encode()).hexdigest()})
            except subprocess.CalledProcessError as error:
                rows.append({"backend": backend, "workers": workers, "status": "failed",
                    "error": f"{error}\n{error.output or error.stderr or ''}"})
    benchmark = pd.DataFrame(rows)
    cpu = benchmark[(benchmark.backend == "local_cpu") & (benchmark.status == "ok")]
    if DETERMINISTIC and len(cpu) and cpu.data_signature.nunique() != 1:
        raise RuntimeError("CPU worker counts produced different deterministic data; investigate before collection")
    duel.write(benchmark_path, {"identity": benchmark_identity, "results": rows})
    display(benchmark.drop(columns="data_signature", errors="ignore"))
    successful = benchmark.loc[benchmark.status.eq("ok")].copy()
    if successful.empty:
        failures = "\n\n".join(f"{row.backend}, {row.workers} workers:\n{row.error}"
                                 for row in benchmark.itertuples())
        raise RuntimeError("Every rollout backend failed. Command diagnostics:\n" + failures)
    best = successful.loc[successful.flights_per_hour.idxmax()]
    BACKEND, WORKERS = str(best.backend), int(best.workers)
    # An existing campaign must keep its recorded inference backend for reproducibility.
    if (WORK / "config.json").exists():
        pinned_backend = duel.read(WORK / "config.json")["inference_backend"]
        if BACKEND != pinned_backend:
            matches = successful.loc[successful.backend.eq(pinned_backend)]
            if matches.empty:
                raise RuntimeError("Existing campaign backend failed its benchmark; use a new campaign to change backend.")
            best = matches.loc[matches.flights_per_hour.idxmax()]
            BACKEND, WORKERS = str(best.backend), int(best.workers)
            print("Resumption retains the existing campaign backend:", BACKEND)
    estimate = duel.estimate_campaign(FLIGHTS, BENCHMARK_FLIGHTS, float(best.wall_seconds),
        float(best.postprocess_seconds), float(best.bytes_per_flight), SHARD_FLIGHTS,
        UPLOAD_MBPS, CHECKSUM_READ_MBPS, GIT_SECONDS_PER_SNAPSHOT)
    display(pd.Series(estimate, name="Estimated full campaign, including LFS upload"))
    print(f"Selected {BACKEND}, {WORKERS} simulator workers.")
    print(f"Planning estimate: {estimate['total_hours']:.1f} hours; sensitivity band "
          f"{estimate['sensitivity_low_hours']:.1f}–{estimate['sensitivity_high_hours']:.1f} hours, "
          f"plus the one-off {len(benchmark)}-candidate benchmark.")
    print(f"Upload assumption {UPLOAD_MBPS:g} Mbps; checksum-read assumption {CHECKSUM_READ_MBPS:g} MB/s. Edit these for your machine.")
    duel.write(WORK / "runtime_estimate.json", {"backend": BACKEND, "workers": WORKERS, **estimate})
    free_gib = shutil.disk_usage(WORK).free/2**30
    if free_gib < 3.5*estimate["payload_gib"]:
        raise RuntimeError("Insufficient free disk for local work, publication and LFS object storage")
else:
    BACKEND = ("central_cuda" if CUDA_AVAILABLE and DETERMINISTIC else "local_cpu") if BACKEND == "auto" else BACKEND
    print("Benchmark disabled; explicitly selected", BACKEND, WORKERS)
run("prepare", backend=BACKEND)
config = duel.read(WORK / "config.json")

## How long should the complete process take?

For the **10 physical core / 16 logical processor Intel machine and RTX 5080 reported
in 09b**, a provisional budget is **4–8 hours end to end**, including collection,
checksums, packaging and LFS uploads, assuming roughly **20 Mbps sustained upload**.
This is a planning range, not a GPU measurement. The old cloud benchmark measured
528 flights/hour, or about **18.9 hours for rollouts alone on that environment**.
A simple eight-effective-worker extrapolation from its 80.75-second serial ten-flight
suite gives about **2.8 hours for acquisition** before storage/network overhead. GPU
offload may reduce that modestly; it cannot accelerate CPU JSBSim physics.

The benchmark above replaces this extrapolation with your machine's measured whole-flight
rate and measured packaging/verification time. It separately budgets repeated prefix
checksum reads, fixed Git overhead and dataset upload at the entered rate. These disk,
network and scaling assumptions are exposed; the sensitivity range is not a statistical
confidence interval. At the measured corpus density, expect roughly **10–15 GiB** of
compressed payload and allow at least **40–55 GiB free** for work, publication and LFS
objects. Every 100-flight cycle then updates an ETA using actual elapsed collection,
verification and upload time. CUDA execution cannot be validated on this CPU-only
cloud environment; the owner's startup parity test and the on-machine benchmark are
required checks before accepting a GPU speedup.

## Collection, restart and off-machine backups

Completed episodes have six checksummed Parquet files (three per perspective) and
an atomic, fsynced completion marker. An interrupted flight is rerun with the same
seed; successful flights are verified and skipped. A campaign lock prevents two
kernels writing the same work directory. After a hard crash, inspect the PID in
`collection.lock` and remove that file only once the process has stopped.

Every 100 new flights the notebook closes immutable shards, verifies published data,
and, when `PUBLISH=True`, commits **only the corpus directory** and pushes this branch.
Git's pre-push hook uploads LFS payloads; publication checks the remote commit afterward.
Partial backups have `complete=false` and explicit physical-flight counts. The final
manifest can say `complete=true` only at 10,000 flights. A failed push raises immediately;
local episode checkpoints remain available to rerun the cell. Unrelated staged changes
are rejected by publication. Existing committed branch work will accompany a push.

Keep the work directory until the final package is verified and its upload is confirmed.
LFS storage and network transfer grow with the measured corpus size. Publication uses
bounded shards below 1 GB, not a single giant file; it does not rewrite old corpora.

The selected CPU/GPU backend is fixed in the campaign configuration. The inference owner is restarted once per backup chunk; its startup is included in the benchmark. Publication verifies each snapshot once and reuses semantic receipts for unchanged episodes. Final verification rereads the entire corpus without a receipt. Live cycle timing updates the remaining-time estimate after packaging and uploads.

### Five-minute progress updates

The notebook prints a short human-readable status about every **300 wall-clock
seconds**, configurable with `PROGRESS_INTERVAL_SECONDS` in the first cell. It
shows saved physical flights and aircraft perspectives, percent complete, elapsed
time, observed flights/hour and ETA, current stage, CPU/GPU backend and worker
count, failed flights in the latest batch, and the most recent verified/LFS-pushed
backup. Benchmark progress is labelled separately. Start, final completion and
errors are reported immediately; backup boundaries do not flood the output.

The same clock spans collection chunks, packaging, verification and LFS uploads,
so long non-rollout stages also get a keepalive. Raw JSON, simulator warnings and
Git output go directly to unique local files in `WORK/notebook_logs/`; the human
updates have their own transcript there. Failure output includes a short log tail.
When a command has not yet completed a flight, the update says so instead of
claiming a stall or inventing an ETA.

Reporting stays in the notebook process: one blocked waiter thread waits for a
command to exit, and a timer wakes the notebook roughly once per five minutes.
Only the existing small `progress.json` counter is read at those intervals.
There are no added worker messages, trajectory scans, checksums, GPU utilization
queries or CUDA synchronizations for reporting. The flight collector and its
source hashes are unchanged, so this reporting update preserves existing campaign
resumption. Final full verification runs as an observed subprocess without a cache;
completion is printed only after that verification succeeds.


In [ ]:
%%time
if COLLECT:
    cycle_start = time.perf_counter()
    start_count = sum(duel.completed(WORK, s, verify=False) is not None for s in config["cases"])
    updates.start_collection(start_count, BACKEND, WORKERS)
    while True:
        config = duel.read(WORK / "config.json")
        count = sum(duel.completed(WORK, s, verify=False) is not None for s in config["cases"])
        if count < FLIGHTS:
            run("collect", workers=WORKERS, backend=BACKEND, limit=min(BACKUP_EVERY, FLIGHTS-count))
        run("package", backend=BACKEND)
        if PUBLISH:
            run("publish", backend=BACKEND)  # Includes checksum and semantic verification.
        else:
            run("verify", backend=BACKEND)
        manifest = duel.read(OUTPUT / "manifest.json")
        completed = manifest["physical_flights"]
        elapsed = time.perf_counter()-cycle_start
        acquired = completed-start_count
        remaining_hours = (FLIGHTS-completed)*elapsed/max(1, acquired)/3600
        updates.snapshot(completed, pushed=PUBLISH)
        duel.write(WORK / "live_eta.json", {"completed": completed, "elapsed_seconds": elapsed,
                                           "estimated_remaining_hours": remaining_hours})
        if manifest["complete"]:
            break
    run("verify", full_verify=True)
    FULLY_VERIFIED = True
    updates.finish()
else:
    run("status")
    print("Collection disabled; this execution did not start the main campaign.")

## Data contract for a later 09b-style experiment

`A01/` stores A01's observation of NN target B01; `B01/` stores B01's observation of
NN target A01. Each has `observations`, `tracks` and privileged `target_decisions`.
`episodes.parquet` identifies target/observer pilots and checkpoint hashes, actual
initial state, shared `scenario_group`, duration and outcome. Parquet row groups are
whole episodes. Time zero and the terminal sample are retained. A terminal sample
has no newly issued decision; future labels are never extended past the final action.

The helper below returns one perspective in the same column contract used by `09b`.
For the A01-observer direction only, its copy of the decision ID is canonicalized to
`A01`, the target name required by `09b.forecast_targets`; `actual_aircraft_id` preserves
physical identity. Observation axes and values are unchanged. This enables the same
**54 causal features**, **51-frame histories plus five acceleration lookback frames**,
first origin at **6 s**, and exact **+1/+2/+5 s** targets with independent missing masks.
The existing `09b.prepare_dataset` hardcodes its old corpus; it does not automatically
consume this new release. A later preparation pass must index these shards explicitly.

Keep **both views of each engagement in the same split**. Fit normalization only on
training histories. Decisions, pilot/checkpoint identity, future tracks and outcomes
remain outside encoder inputs. Combining two observations into 108 inputs is a
separate omniscient task unless both streams are available at inference time.
Reserve validation before using skill-gap feedback to tune scenario distributions;
actively selected follow-up data belongs in training, with a separate fixed evaluation
suite. Catalogue labels can share executed control code and are not physical-skill proof.

In [ ]:
%%time
if (OUTPUT / "manifest.json").exists():
    if not FULLY_VERIFIED:
        run("verify")
    manifest = duel.read(OUTPUT / "manifest.json")
    episodes = pd.read_parquet(OUTPUT / "episodes.parquet")
    display(episodes.groupby(["observer_id", "family"]).agg(
        perspectives=("episode_id", "size"), mean_duration_s=("duration_s", "mean")))
    if len(episodes):
        episode = episodes.iloc[0]
        observations, tracks, target_decisions = duel.load_forecasting_view(
            OUTPUT, episode.episode_id, episode.observer_id)
        from bvr_behavior_prediction.paired_skill_forecasting import trained_examples
        classes = duel.read(WORK / "tactical_snapshot/training/pilot_001/config.json")["skill_names"]
        features, origins, targets = trained_examples(observations, tracks, target_decisions, classes)
        display(pd.Series({"physical_flights": manifest["physical_flights"],
            "perspectives": manifest["perspective_episodes"], "example_feature_shape": features.shape,
            "forecast_origins": len(origins), "valid_labels_per_horizon": (targets >= 0).sum(axis=0).tolist()}))
        display(target_decisions.head())
else:
    print("No collection snapshot yet.")

## Where might the missing skills appear? Hypotheses to test

The saved pre-split survey in `09b` on this branch examined 3,000 NN-versus-scripted
flights. **`commit`, `dive_defense`, and `drag_missile` had zero eligible episodes at
all three forecast horizons.** `accelerate_to_speed` and `climb_to_altitude` had only
**two episodes** each; `lock_target` had four. These are archived observations, not a
promise that a larger corpus will make those classes appear. The new audit below
uses the same six-second eligibility and exact-horizon rules and retains zero classes.

* **Energy/altitude gaps:** low-energy starts with a higher opponent may encourage
  acceleration and climbing; inverted height advantages may encourage descent and
  vertical defence. The low-energy/climb and high/low families test this. Full-throttle
  bypass can make acceleration choices physically indistinguishable; changing it
  requires a separately versioned environment and likely new pilot adaptation.
* **Commit/recommit:** cold, crossing and tail-chase starts place targets behind or
  outside the initial radar cone and may create search, lock and re-engagement
  transitions. Offset/support starts vary missile support geometry. Prolonging flights
  allows more post-shot/recommit phases, but first defeat still terminates the episode.
* **Missile defence gaps:** close merges and defensive beam starts may elicit earlier
  opposing launches and change time-to-impact and aspect, possibly yielding dive,
  drag or last-ditch responses. Recording incoming missiles and remaining weapons
  helps distinguish lack of threat exposure from refusal to select a defensive skill.
* **Adaptive next batch (speculation, not implemented):** after an initial development
  batch, relate eligible skill counts to family, energy gap, aspect, range, pilot pair
  and missile threat exposure. Sample undercovered cells and perturb promising starts;
  use Bayesian optimization or a cross-entropy search over initial conditions with
  rare-skill episode support as its score. Keep a broad random component, log sampling
  probabilities and seeds, and avoid success-only filtering. Count independent scenario
  groups and transitions, not just thousands of overlapping windows.
* **Policy support limits:** a frozen deterministic pilot may never choose a catalogue
  entry anywhere reachable. Inspect its logits on diverse histories first. A separately
  labelled stochastic-policy campaign (`DETERMINISTIC=False`, fresh work/output) can
  sample lower-probability choices while preserving NN control; it changes the behaviour
  distribution. If the missing skills have essentially zero probability, use curriculum
  fine-tuning or skill-conditioned demonstrators in a separate study. Forced catalogue
  actions would not demonstrate that these frozen pilots independently chose the skill.

Scenario novelty, catalogue coverage, physical behaviour and forecast learnability
are four distinct measurements. Report both sides and pilot-specific counts before
claiming a gap has been filled; a zero may persist despite all 10,000 new flights.

In [ ]:
%%time
if (OUTPUT / "skill_support.parquet").exists():
    support = pd.read_parquet(OUTPUT / "skill_support.parquet")
    display(support.pivot(index="skill", columns="horizon_s", values="windows"))
    display(support.pivot(index="skill", columns="horizon_s", values="perspective_episodes"))
    print("Still missing at one or more horizons:")
    display(support[support.windows.eq(0)])
    breakdown = pd.read_parquet(OUTPUT / "skill_support_by_pilot_family.parquet")
    display(breakdown.groupby(["pilot_id", "horizon_s"]).agg(supported_skills=("skill", lambda s: s[breakdown.loc[s.index, "windows"].gt(0)].nunique())))
    print("These episode counts include two correlated perspectives per physical engagement; split by scenario_group.")
else:
    print("Skill support will be measured from actual selections after the first closed shard.")